Conversation Management & Classification using Groq API

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install groq

import os
from groq import Groq


os.environ["GROQ_API_KEY"] = "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"

client = Groq()


response = client.models.list()
print([m.id for m in response.data])


['gemma2-9b-it', 'playai-tts', 'llama-3.1-8b-instant', 'meta-llama/llama-prompt-guard-2-86m', 'openai/gpt-oss-20b', 'allam-2-7b', 'whisper-large-v3-turbo', 'llama-3.3-70b-versatile', 'groq/compound', 'moonshotai/kimi-k2-instruct-0905', 'moonshotai/kimi-k2-instruct', 'groq/compound-mini', 'meta-llama/llama-4-scout-17b-16e-instruct', 'meta-llama/llama-prompt-guard-2-22m', 'whisper-large-v3', 'playai-tts-arabic', 'openai/gpt-oss-120b', 'qwen/qwen3-32b', 'deepseek-r1-distill-llama-70b', 'meta-llama/llama-guard-4-12b', 'meta-llama/llama-4-maverick-17b-128e-instruct']


# Task 1: Conversation Management with Summarization


In [ ]:
#ConversationManager class
class ConversationManager:
    def __init__(self, k=3):
        self.history = []
        self.run_count = 0
        self.k = k

    def add_message(self, role, content):
        """Add a message and trigger summarization every k-th run"""
        self.history.append({"role": role, "content": content})
        self.run_count += 1
        if self.run_count % self.k == 0:
            self.summarize_history()

    def truncate_by_turns(self, n):
        """Return only the last n messages"""
        return self.history[-n:]

    def truncate_by_length(self, max_chars):
        """Return messages up to a maximum character length"""
        text = ""
        truncated = []
        for msg in reversed(self.history):
            if len(text) + len(msg["content"]) <= max_chars:
                truncated.insert(0, msg)
                text += msg["content"]
            else:
                break
        return truncated

    def summarize_history(self):
        """Summarize the conversation so far using Groq API"""
        summary_prompt = "Summarize this conversation briefly:\n" + "\n".join(
            [f"{m['role']}: {m['content']}" for m in self.history]
        )
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": summary_prompt}]
        )
        summary = response.choices[0].message.content

        self.history = [{"role": "system", "content": f"Summary so far: {summary}"}]


In [ ]:
# Create conversation manager (summarize every 3rd message)
cm = ConversationManager(k=3)


cm.add_message("user", "Hi, I want to book a cab.")
cm.add_message("assistant", "Sure, where do you want to go?")
cm.add_message("user", "To the airport please.")

print("After 3rd run (summarized):")
print(cm.history)


cm.add_message("assistant", "Got it, when do you want to leave?")
cm.add_message("user", "Tomorrow morning.")

print("\nTruncated by turns (last 2):")
print(cm.truncate_by_turns(2))

print("\nTruncated by chars (50):")
print(cm.truncate_by_length(50))


After 3rd run (summarized):
[{'role': 'system', 'content': 'Summary so far: This conversation has just started, and you asked to book a cab to the airport. It has not yet been responded to by the AI.'}]

Truncated by turns (last 2):
[{'role': 'assistant', 'content': 'Got it, when do you want to leave?'}, {'role': 'user', 'content': 'Tomorrow morning.'}]

Truncated by chars (50):
[{'role': 'user', 'content': 'Tomorrow morning.'}]


Task 2 (JSON Schema Classification & Extraction)

JSON Schema Classification & Information Extraction

In [ ]:
model="openai/gpt-oss-120b"


In [ ]:
from groq import Groq
import os
import json


client = Groq(api_key=os.environ["GROQ_API_KEY"])


functions = [
    {
        "name": "extract_user_info",
        "description": "Extract user details from chat",
        "parameters": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "email": {"type": "string"},
                "phone": {"type": "string"},
                "location": {"type": "string"},
                "age": {"type": "integer"},
            },
            "required": ["name", "email", "phone", "location", "age"],
        },
    }
]

def extract_info_from_chat(chat_text):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": chat_text}],
        functions=functions,
        function_call={"name": "extract_user_info"},
    )
    return json.loads(response.choices[0].message.function_call.arguments)


samples = [
    "Hi, I'm Alice, 25 years old. My email is alice@example.com and my phone is 9876543210. I live in Mumbai.",
    "Hello, my name is Bob. I'm 30. You can reach me at bob@gmail.com, phone 9123456789. I'm from Delhi.",
    "Hey there, it's Carol, 28 years old, email carol@outlook.com, number 9988776655, living in Bangalore."
]

for i, chat in enumerate(samples, 1):
    print(f"🔹 Chat {i}: {chat}")
    extracted = extract_info_from_chat(chat)
    print(" Extracted JSON:", json.dumps(extracted, indent=2))
    print("-" * 60)


🔹 Chat 1: Hi, I'm Alice, 25 years old. My email is alice@example.com and my phone is 9876543210. I live in Mumbai.
 Extracted JSON: {
  "age": 25,
  "email": "alice@example.com",
  "location": "Mumbai",
  "name": "Alice",
  "phone": "9876543210"
}
------------------------------------------------------------
🔹 Chat 2: Hello, my name is Bob. I'm 30. You can reach me at bob@gmail.com, phone 9123456789. I'm from Delhi.
 Extracted JSON: {
  "age": 30,
  "email": "bob@gmail.com",
  "location": "Delhi",
  "name": "Bob",
  "phone": "9123456789"
}
------------------------------------------------------------
🔹 Chat 3: Hey there, it's Carol, 28 years old, email carol@outlook.com, number 9988776655, living in Bangalore.
 Extracted JSON: {
  "age": 28,
  "email": "carol@outlook.com",
  "location": "Bangalore",
  "name": "Carol",
  "phone": "9988776655"
}
------------------------------------------------------------


In [ ]:
!pip install jsonschema

from jsonschema import validate, ValidationError

# JSON Schema
schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "email": {"type": "string"},
        "phone": {"type": "string"},
        "location": {"type": "string"},
        "age": {"type": "integer"},
    },
    "required": ["name", "email", "phone", "location", "age"],
}


for i, chat in enumerate(samples, 1):
    extracted = extract_info_from_chat(chat)
    print(f"🔹 Chat {i} Extracted JSON:", extracted)
    try:
        validate(instance=extracted, schema=schema)
        print("Validation Passed")
    except ValidationError as e:
        print("Validation Failed:", e.message)
    print("-" * 60)


🔹 Chat 1 Extracted JSON: {'name': 'Alice', 'email': 'alice@example.com', 'phone': '9876543210', 'location': 'Mumbai', 'age': 25}
Validation Passed
------------------------------------------------------------
🔹 Chat 2 Extracted JSON: {'name': 'Bob', 'email': 'bob@gmail.com', 'phone': '9123456789', 'location': 'Delhi', 'age': 30}
Validation Passed
------------------------------------------------------------
🔹 Chat 3 Extracted JSON: {'name': 'Carol', 'email': 'carol@outlook.com', 'phone': '9988776655', 'location': 'Bangalore', 'age': 28}
Validation Passed
------------------------------------------------------------
